In [1]:
import numpy as np
from dolfinx import default_scalar_type
from dolfinx.fem import (
    Constant,
    Function,
    functionspace,
    dirichletbc,
    locate_dofs_geometrical,
    form,
    extract_function_spaces,
)
from dolfinx.fem.petsc import (
    LinearProblem, 
    NonlinearProblem, 
    create_matrix, 
    create_vector, 
    assemble_matrix, 
    assemble_vector,
    apply_lifting,
    set_bc,
    )
from dolfinx.mesh import create_unit_square, locate_entities, meshtags, create_rectangle, CellType
from dolfinx.plot import vtk_mesh

from mpi4py import MPI
from ufl import (
    SpatialCoordinate,
    TestFunction,
    TrialFunction,
    dx,
    grad,
    div,
    inner,
    exp,
    lhs,
    rhs,
    derivative,
    conditional,
    sqrt,
    ln,
)

from petsc4py import PETSc
from visualization_fct import *


In [3]:
# Define time discretization parameters
t = 0.0 # start time
T = 60*60 # end time
delta_t = 1e-2 # time step size
num_steps = int(np.floor(T/delta_t))
print(num_steps)

# Define constants
alpha = 14.5
N = 2.68
theta_r = 0.045
theta_s = 0.43
Ks = 8.25e-5

L = a = 15.24 # height, length of box
h_r = -15.24 # pressure head when the soil is very dry

# Solution constants
h_0 = 1 - np.exp(alpha*h_r)
beta = np.sqrt(alpha**2/4 + (np.pi/a)**2)
beta_1 = np.sqrt(alpha**2/4 + (np.pi*2/a)**2)

360000


In [9]:
# Create mesh
domain = create_rectangle(MPI.COMM_WORLD, ((0,0), (a, L)), (100,100), cell_type=CellType.quadrilateral)
V = functionspace(domain, ("CG", 1)) # linear Lagrange
x = SpatialCoordinate(domain)

def Se(u):
    eps = 1e-12
    m = 1 - 1/N
    S = conditional(u < -eps, (1 + (alpha * abs(u))**N)**(-m), 1)
    return S

def theta(u):
    return conditional(u < 0, theta_r + (theta_s - theta_r)*Se(u), theta_s)

def k(u):
    m = 1 - 1/N
    S = Se(u)
    kr = conditional(S < 1, sqrt(S) * (1 - (1 - S**(1/m))**m)**2, 1)
    return kr * Ks

def C(u):
    m = 1 - 1/N
    Sm = Se(u)**(1/m)
    Cval = conditional(Sm < 1, alpha * m * (theta_s - theta_r) * N * Sm * (1 - Sm)**m, 0)
    return Cval


In [10]:
# Define Dirichlet boundary conditions

def left_or_right(x):
    return np.logical_or(np.isclose(x[0], 0), np.isclose(x[0], a))

dofs_fix = locate_dofs_geometrical(
    V, lambda x: np.logical_or(left_or_right(x), np.isclose(x[1], 0)))
bc_fix = dirichletbc(Constant(domain, default_scalar_type(h_r)), dofs_fix, V)



def expr_h_fix(x):
    return 1/0.19*np.log(np.exp(0.19*h_r) + h_0*np.sin(np.pi*x[0]/a))

dofs_L = locate_dofs_geometrical(V, lambda x: np.isclose(x[1], L))

h_top_fix = Function(V)
h_top_fix.interpolate(expr_h_fix)
bc_top_fix = dirichletbc(h_top_fix, dofs_L)

In [11]:
v = TestFunction(V)
uh = Function(V)
u_N = Function(V)
u_N.x.array[:] = h_r # initial value
x = SpatialCoordinate(domain)
bc = [bc_fix, bc_top_fix]

F = C(uh) / delta_t * (uh - u_N) * v * dx
F += inner(k(uh) * grad(x[1] + uh), grad(v)) * dx

J = derivative(F, uh)

petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "bt",
    "snes_atol": 1e-10,
    "snes_rtol": 1e-10,
    "snes_monitor_cancel": None,
    "ksp_error_if_not_converged": True,
    "ksp_type": "preonly",
    "ksp_rtol": 1e-10,
    "ksp_monitor_cancel": None,
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}

for n in range(num_steps):
    problem = NonlinearProblem(
        F,
        uh,
        bcs=bc,
        J=J,
        petsc_options=petsc_options,
        petsc_options_prefix="SNES_",
    )

    uh = problem.solve()
    converged = problem.solver.getConvergedReason()
    num_iter = problem.solver.getIterationNumber()
    assert converged > 0, f"Solver did not converge, got {converged}."
    """ print(
        f"Solver converged at timestep {n} after {num_iter} iterations with converged reason {converged}."
    ) """

    u_N.x.array[:] = uh.x.array

AssertionError: Solver did not converge, got -6.